# Racialised language in German swimming-pool reviews
### Preliminary quantitative findings toward an ICWSM submission — discussion draft, iteration 1

Ethel Mensah · ICWSM prep meeting · 14 July 2026

**Framing for today**: these are first-pass, preliminary quantitative findings — meant to show a
working, iterable analysis process, not a finished result. The goal of this meeting is to agree
what's solid enough to build on and what still needs work, not to defend a final number.

*This notebook is deliberately short. It is the handful of findings from the fuller notebooks
(`01_gmaps_feasibility`, `02_socioeconomic_political_relations`, `03_predictive_modelling`)
that are simple enough to present, explain, and defend in a 10-15 minute discussion.*

## The question, and how the analysis was built up

Do Google Maps reviews of German swimming pools ever describe experiences in racialised
terms — and if so, is that spread evenly across the country, or concentrated somewhere?

The analysis was built in stages, and this notebook follows the same stages:

- **Part A** — what's quantitatively feasible using the review data alone
- **Part B** — linking that to external socioeconomic + political data, constituency by constituency
- **Part C** — a first check of whether the Part B signal survives being tested more rigorously, all at once

Each part only builds on what the previous part already established.

## What's in the data

- **1,265,806 reviews** of **5,529 swimming-pool venues** across Germany (Google Maps)
- Every review was run through a classifier that flags whether it (a) describes a
  **social interaction** at all, and (b) whether that description is **racialised**
  (mentions race, ethnicity, or nationality)
- We restrict to the ~504,000 reviews that describe a social interaction, and ask what
  share of those are racialised

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

BASE = "/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026"
# Every chart below saves a copy here as a plain PNG, numbered in slide order -- use these
# directly when building slides (PowerPoint/Google Slides/Keynote), rather than screenshotting.
FIGDIR = f"{BASE}/figures/presentation_slides"
os.makedirs(FIGDIR, exist_ok=True)

const_agg = pd.read_parquet(f"{BASE}/notebook_outputs/const_agg.parquet")

total_social = const_agg["social_reviews"].sum()
total_racialised = const_agg["racialised_reviews"].sum()
overall_rate = total_racialised / total_social

print(f"Constituencies covered:      {len(const_agg)}")
print(f"Social reviews analysed:     {total_social:,}")
print(f"...of which racialised:      {total_racialised:,}")
print(f"Overall racialised rate:     {overall_rate:.2%}")

**Headline number: about 1 in every 65 social reviews (1.5%) is racialised.** Part A below
is about how that number moves on its own — over time and across the country — using
nothing but the review data itself.

---
# Part A — What the review data alone can tell us

## A1. It's becoming more common

Restricting to reviews we could date (recovering the year from Google's "2 years ago"-style
timestamps where an exact date wasn't available), we can trace the rate year by year.

In [ ]:
import re

cols = ["social", "racialised", "review_timestamp", "review_time", "place_retrieved_time", "unique_identifier"]
corpus = pd.read_parquet(f"{BASE}/gmaps_data/master_corpus_final.parquet", columns=cols)
social = corpus.loc[corpus["social"] == "yes"].copy()

# Most reviews already have an exact timestamp. The rest are missing one because the source
# DB needs a scrape date to convert Google's relative text ("2 years ago") into an absolute
# date, and that scrape date (`retrieved_time`) is itself null for ~99.8% of these rows -- so
# it's a missing-reference-date problem, not random/unexplained missingness. We convert these
# using the *place's* scrape date instead (`place_retrieved_time`, never null) as a stand-in
# reference point. Coarse (year-level), but fine for a yearly trend.
REL = re.compile(r"^(?:Edited\s+)?(a|an|\d+)\s+(year|month|week|day|hour|minute)s?\s+ago$", re.IGNORECASE)
DAYS = {"year": 365.25, "month": 30.44, "week": 7, "day": 1, "hour": 1/24, "minute": 1/(24*60)}

def parse_relative(text):
    m = REL.match(text) if isinstance(text, str) else None
    if not m:
        return None
    qty_raw, unit = m.groups()
    qty = 1 if qty_raw.lower() in ("a", "an") else int(qty_raw)
    return pd.Timedelta(days=qty * DAYS[unit.lower()])

missing = social["review_timestamp"].isna()
deltas = social.loc[missing, "review_time"].apply(parse_relative)

base_ts = social["review_timestamp"].dt.tz_localize(None).astype("datetime64[us]")
est = pd.Series(pd.NaT, index=social.index, dtype="datetime64[us]")
est.loc[missing] = (social.loc[missing, "place_retrieved_time"] - deltas).astype("datetime64[us]")

# Keep both versions of the year: the exact-timestamp-only one, and the one that also
# uses the recovered/estimated dates -- so we can check the trend isn't just an artifact
# of the recovery step.
social["year_actual"] = base_ts.dt.year
social["year_combined"] = base_ts.fillna(est).dt.year

def yearly_rate(col):
    yearly = (
        social.dropna(subset=[col])
        .groupby(col)
        .agg(social_reviews=("racialised", "size"),
             racialised_reviews=("racialised", lambda s: (s == "yes").sum()))
    )
    yearly["rate"] = yearly["racialised_reviews"] / yearly["social_reviews"]
    # Keep 2021-2025: platform use before ~2019 is too sparse/noisy to read a trend from
    # (few hundred reviews some years), and 2026 is only a partial year so far.
    return yearly.loc[2021:2025]

yearly_combined = yearly_rate("year_combined")
yearly_actual = yearly_rate("year_actual")
n_places_missing = social.loc[missing, "unique_identifier"].nunique()
print("actual-timestamp coverage:", f"{base_ts.notna().mean():.1%}")
print(f"places (pools) with at least one estimated-date review: {n_places_missing:,} of {social['unique_identifier'].nunique():,}")
print()
print(yearly_combined)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(yearly_combined.index, yearly_combined["rate"] * 100, marker="o", color="#C44E52", linewidth=2)
for yr, val in zip(yearly_combined.index, yearly_combined["rate"] * 100):
    ax.annotate(f"{val:.2f}%", (yr, val), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, fontweight="bold")
ax.set_ylabel("racialised reviews, as % of social reviews\nthat year (German swimming-pool reviews)")
ax.set_xlabel("year")
ax.set_title("Racialised rate by year, 2021-2025\n(N=299 constituencies, 504K social reviews)")
ax.set_xticks(yearly_combined.index)
ax.set_ylim(0, yearly_combined["rate"].max() * 100 * 1.25)  # headroom so labels don't clip
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
fig.savefig(f"{FIGDIR}/01_trend_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it simply**: the share roughly doubled from 2021 to 2025 (about 0.9% to about
2.1%), rising every year over that window. Years before 2021 are excluded here — reviews
are too sparse before the platform saw wider use to read a trend from, and 2026 is only a
partial year so far.

### Robustness check — does this hold without the estimated dates?

About 30% of the dates above (152,604 reviews) come from converting relative text like
"2 years ago" into an estimated date rather than an exact timestamp — we don't have the
original raw scrape files to look up an exact date for these specific rows, so estimating
from the place's scrape date is the only way to include them at all, rather than dropping
them. Those estimated-date reviews come from 2,254 of the corpus's 5,305 pools (42%) — not
a small, unusual subset of venues. Worth checking directly: does the trend survive if we
throw all of them out and only use the ~70% of reviews with an exact timestamp?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(yearly_combined.index, yearly_combined["rate"] * 100, marker="o",
        color="#C44E52", linewidth=2, label="all reviews (exact + estimated dates)")
ax.plot(yearly_actual.index, yearly_actual["rate"] * 100, marker="s", linestyle="--",
        color="#4C72B0", linewidth=2, label="only reviews with an exact timestamp (~70%)")
for yr, val in zip(yearly_combined.index, yearly_combined["rate"] * 100):
    ax.annotate(f"{val:.2f}%", (yr, val), textcoords="offset points", xytext=(0, -16),
                ha="center", fontsize=8, color="#C44E52", fontweight="bold")
for yr, val in zip(yearly_actual.index, yearly_actual["rate"] * 100):
    ax.annotate(f"{val:.2f}%", (yr, val), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=8, color="#4C72B0", fontweight="bold")
ax.set_ylabel("racialised reviews, as % of social reviews\nthat year (German swimming-pool reviews)")
ax.set_xlabel("year")
ax.set_title("Racialised rate by year, 2021-2025:\nrises whether or not estimated dates are included")
ax.set_xticks(yearly_combined.index)
ax.set_ylim(0, yearly_actual["rate"].max() * 100 * 1.3)  # headroom so labels don't clip
ax.legend(fontsize=9, loc="upper left")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
fig.savefig(f"{FIGDIR}/02_trend_robustness.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it simply**: both lines rise the same way, year over year — if anything the
exact-timestamp-only line rises slightly faster (0.8% in 2021 to 2.3% in 2025). The
trend isn't an artifact of how we filled in missing dates, or of which venues happen to have
them.

## A2. It's not spread evenly across the country

Same rate, mapped by constituency instead of by year.

In [ ]:
from PIL import Image

img = Image.open(f"{BASE}/figures/presentation_map_crop.png")
fig, ax = plt.subplots(figsize=(6, 6.5))
ax.imshow(img)
ax.axis("off")
ax.set_title("Racialised rate by German federal constituency (N=299)\n(darker = higher; grey/hatched = too few reviews yet to trust a rate)", fontsize=11)
plt.tight_layout()
fig.savefig(f"{FIGDIR}/03_map.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it simply**: darker red = a higher share of racialised reviews. A handful of
constituencies stand out consistently — parts of the Ruhr region, Berlin, and the southwest
(Offenburg, Lörrach) — rather than the colour being scattered randomly across the map.
That clustering is itself a finding worth discussing, separate from the overall rate.

## A3. Ruling out the obvious objection

The natural pushback: *"couldn't a high rate just be one bad pool dragging up a whole
constituency's number?"* Two complementary checks, shown side by side: (left) what share of
a constituency's pools have **any** racialised review at all, and (right) how much of a
constituency's racialised reviews sit at just its single most-implicated pool.

In [ ]:
outlier_mask = const_agg["dominant_pool_is_small_high_rate"]

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
fig.suptitle("Is a constituency's racialised rate driven by many pools, or just one?\n"
             "(299 German federal constituencies; same y-axis, constituency's racialised rate, on both panels)",
             fontsize=11)

panel_specs = [
    ("pct_pools_with_racialised", axes[0], "% of a constituency's pools with >=1 racialised review",
     "Share of pools affected\n\u2192 tracks the constituency's racialised rate closely"),
    ("top_pool_share_of_racialised", axes[1], "% of a constituency's racialised reviews at its single top pool",
     "Top-pool dominance\n\u2192 does NOT track the constituency's racialised rate"),
]
for xcol, ax, xlabel, title in panel_specs:
    ax.scatter(const_agg.loc[~outlier_mask, xcol] * 100, const_agg.loc[~outlier_mask, "racialised_rate"] * 100,
               alpha=0.5, color="#4C72B0", edgecolor="none", label="most constituencies (280 of 299)")
    ax.scatter(const_agg.loc[outlier_mask, xcol] * 100, const_agg.loc[outlier_mask, "racialised_rate"] * 100,
               alpha=0.9, color="#DD8452", edgecolor="black", linewidth=0.5, marker="D", s=45,
               label="one pool drives most of it (19 of 299)")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("constituency's racialised rate (%)")
    ax.tick_params(labelleft=True)  # keep y-axis numbers on both panels despite sharey (each must stand alone)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

axes[0].legend(fontsize=8, loc="upper left")
plt.tight_layout()
fig.savefig(f"{FIGDIR}/04_venue_concentration.png", dpi=150, bbox_inches="tight")
plt.show()

print(const_agg.loc[outlier_mask, ["constituency", "state", "racialised_rate"]]
      .sort_values("racialised_rate", ascending=False).head(5).to_string(index=False))

**Reading the left panel**: the two move together — constituencies with a higher overall
rate are ones where *more* of their pools carry at least one flagged review, not
constituencies dominated by a single bad venue. On average only about 4 in 10 pools in a
constituency have any flagged review, but that share tracks the constituency's rate closely.

**Reading the right panel**: if a high rate were really just one bad pool, this panel would
show the same upward slope as the left one — it doesn't. How concentrated a constituency's
racialised reviews are in its single top pool has **no relationship** with how high its
overall rate is. Together, both panels say the same thing: this reads as a broad,
constituency-wide pattern rather than a one-venue artifact — which is what makes the
geography in A2 worth taking seriously.

**The honest exception, shown rather than hidden**: in 19 of 299 constituencies (6%, orange
diamonds — visible on the left mostly at low-to-mid values, and on the right at
high values, exactly where you'd expect a genuinely single-venue-driven case to sit), one
single pool really does account for a disproportionate share of that constituency's
racialised reviews — led by `Dortmund II` and `Krefeld I – Neuss II`. Worth naming directly
if asked, but their own rates aren't the extreme ones on the map in A2, and they're a small
minority — they don't change the broad pattern above.

## Part A conclusion — what's solid with the review data alone

Three things hold up using nothing but the reviews themselves:

1. A defensible headline prevalence estimate (1.5%) that is **rising over time** (A1),
   confirmed even when we throw out the estimated dates.
2. **Real geographic clustering** (A2) — specific constituencies stand out consistently,
   not random scatter.
3. That clustering is a **broad, constituency-level pattern**, not a handful of bad venues
   wearing a constituency's name (A3) — with a small, transparently-flagged exception.

That's already enough to support a descriptive "where and when does this concentrate"
story on its own. Part B asks whether we can also start explaining *why*.

**A note on vocabulary, in case it comes up**: Part A above is quantitative analysis — real
numbers, rates, trends, and a statistically-confirmed spatial pattern — it's just
**descriptive** quantitative analysis (measuring what's there) rather than **explanatory**
(why it's there). That's a legitimate, standard category of quantitative work on its own,
not a lesser or "not really quantitative" stand-in for it. Part B moves into
**correlational** analysis (does the rate track other measured things), and Part C into a
first **predictive** model (do those things explain it jointly). All three are quantitative;
they just answer different questions.

---
# Part B — Linking to socioeconomic + political data

Everything above uses only the review data. For Part B, we joined each constituency's rate
to public German census/election data for that same constituency (population, income,
unemployment, benefits recipients, vote shares) and asked: does the rate track any of
these?

In [ ]:
from scipy import stats as sci_stats

political = pd.read_csv(f"{BASE}/socio-economic_data/federal_baseline_eng_selected_political.csv")
political_extra = political.drop(columns=["state", "constituency"])
analysis_df = const_agg.merge(political_extra, on="constituency_code", how="left")
analysis_df["foreign_population_pct"] = analysis_df["foreign_population"] / analysis_df["total_population"] * 100

quartile_bins = {}
for col in ["foreign_population_pct", "second_vote_afd_pct"]:
    analysis_df[f"{col}_quartile"], bins = pd.qcut(analysis_df[col], 4, labels=["Q1", "Q2", "Q3", "Q4"], retbins=True)
    quartile_bins[col] = bins

print(f"Merged {len(analysis_df)} constituencies with socioeconomic + political data.")

### We didn't just check these two — 18 measures were tested

Foreign-population share and AfD vote share aren't cherry-picked; they're the two clearest
results from a full sweep. `02_socioeconomic_political_relations.ipynb` section 2.4 tested
**18** socioeconomic/political measures against the rate one at a time — age structure,
income, GDP, employment, every major party's vote share, turnout, and more. At first glance,
**17 of 18 looked related** to the rate. But section 2.4b then checked all 18 against how
many pools a constituency has (the same pool-density check from A3/B2) — and **most of them
washed out**. Only the deprivation cluster (foreign-population share, benefits-recipients,
unemployment) and turnout survived. That washout — lots of plausible-looking candidates, few
that survive scrutiny — is exactly what motivated building one proper predictive model that
tests everything together at once, rather than one at a time (that model is Part C, below).

In [ ]:
import numpy as np
from scipy.stats import rankdata

def partial_spearman(x, y, control):
    x, y, control = np.asarray(x, float), np.asarray(y, float), np.asarray(control, float)
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(control)
    A = np.vstack([rz, np.ones_like(rz)]).T
    bx, *_ = np.linalg.lstsq(A, rx, rcond=None)
    by, *_ = np.linalg.lstsq(A, ry, rcond=None)
    return sci_stats.pearsonr(rx - A @ bx, ry - A @ by)

analysis_df["sgb_ii_recipients_pct"] = analysis_df["sgb_ii_recipients_total"] / analysis_df["total_population"] * 100

survivor_predictors = [
    ("sgb_ii_recipients_pct", "% receiving basic benefits"),
    ("foreign_population_pct", "% foreign-born population"),
    ("unemployment_total", "unemployment rate"),
    ("turnout_pct", "voter turnout"),
    ("second_vote_spd_pct", "SPD vote share"),
    ("second_vote_bsw_pct", "BSW vote share"),
    ("second_vote_afd_pct", "AfD vote share"),
]

rows = []
for col, label in survivor_predictors:
    rho_raw, _ = sci_stats.spearmanr(analysis_df[col], analysis_df["racialised_rate"])
    rho_adj, _ = partial_spearman(analysis_df[col], analysis_df["racialised_rate"], analysis_df["pool_count"])
    rows.append((label, rho_raw, rho_adj))

rows.sort(key=lambda r: abs(r[1]))  # smallest raw effect at the bottom, largest at the top
labels = [r[0] for r in rows]

fig, ax = plt.subplots(figsize=(7.5, 5))
for i, (label, rho_raw, rho_adj) in enumerate(rows):
    ax.plot([rho_raw, rho_adj], [i, i], color="#999999", linewidth=1.5, zorder=1)
ax.scatter([r[1] for r in rows], range(len(rows)), color="#DD8452", s=70, zorder=2, label="raw correlation")
ax.scatter([r[2] for r in rows], range(len(rows)), color="#4C72B0", s=70, zorder=2, label="after accounting for pool count")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels(labels)
ax.set_xlabel("correlation with constituency's racialised rate (-1 to +1; 0 = no relationship)")
ax.set_title("7 of 18 socioeconomic/political measures tested (299 German constituencies):\nwhat survives accounting for how many pools a constituency has")
ax.legend(fontsize=8, loc="lower right")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
fig.savefig(f"{FIGDIR}/04b_correlation_survivors.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it simply**: each row is one factor tested against the rate. The orange dot is
the raw correlation; the blue dot is the same correlation after accounting for pool count
(how many pools a constituency has, which affects almost everything on this list). A short
grey line barely moving = a real, if weaker, relationship. A long grey line swinging past
zero = the AfD case from B2 — it doesn't just weaken, it disappears (crosses from clearly
negative to essentially nothing). The three deprivation measures at the top (benefits,
foreign-population share, unemployment) and turnout are the only ones that stay meaningfully
away from zero after this check — everything else on this list, and the 11 other measures
not shown here, either shrink to nothing or were never significant to begin with.

## B1 & B2. Two checks against local demographics — shown side by side

Same method for both: split all 299 constituencies into four equal-sized groups (quartiles)
by a demographic measure, and compare each group's typical (median) racialised rate. Showing
both together so they can be compared directly, the way they were in the source notebook
(`02_socioeconomic_political_relations.ipynb`, §2.5).

**What's already been accounted for, in both charts below:**
- Each group has ~75 constituencies — not a handful of unusual places driving the bar.
- Checked separately with the 11 lowest-data-volume constituencies removed — the pattern
  doesn't change (`02_socioeconomic_political_relations.ipynb`, §2.4).
- Checked against how many pools a constituency has (venue density affects the rate on its
  own) — noted per chart below where it does/doesn't survive that check.
- Tested for statistical significance with a rank-based group-comparison test (Kruskal-Wallis,
  appropriate here because the rate is a small, skewed proportion, not normally distributed)
  — p-values reported below the charts, not left unstated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
fig.suptitle("299 German constituencies split into 4 equal-sized groups (quartiles);\n"
             "bars show each group's typical (median) racialised rate", fontsize=11)
---
# Research questions — what's answered, and what's next

Three candidate research questions came out of this analysis (see
`02_socioeconomic_political_relations.ipynb` section 2.8 for the original framing). Stating
them explicitly, with an honest status for each:

**RQ1 — Where and when does racialised discourse concentrate, and is that spatially
structured or incidental?**
**Status: answered, ready to write up close to as-is.** This is Part A: a real temporal rise
(A1), genuine geographic clustering confirmed statistically, not just visually (A2), and
confirmed to be a broad area pattern rather than a venue artifact (A3).

**RQ2 — What structural (socioeconomic/demographic) features of a place are associated with
this, net of confounding between those features (including venue density itself)?**
**Status: first pass done, real but incomplete.** Part B/C: foreign-born population share is
the one predictor that survives every check we've run, including a joint model with
everything else accounted for at once. Still open: untangling the two deprivation measures
that are too collinear to separate, and explaining the geographic clustering this model
still doesn't fully account for.

**RQ3 — Does the *content* of racialised discourse (not just how often it occurs) vary
systematically by place?**
**Status: not covered in this notebook, kept out to stay focused today.** An early look (in
`02_socioeconomic_political_relations.ipynb` section 2.7, not presented here) suggests the
themes reviews raise shift modestly by region while their overall tone doesn't — worth a
follow-up slide once the underlying `theme`/`approach` labels have been through inter-annotator
agreement checks, which isn't finished yet.
def labeled_ticks(col):
    bins = quartile_bins[col]
    n_per_group = analysis_df[f"{col}_quartile"].value_counts().sort_index()
    labels = ["Q1", "Q2", "Q3", "Q4"]
    return [f"{labels[i]}: {bins[i]:.1f}-{bins[i+1]:.1f}%\n(n={n_per_group.iloc[i]})" for i in range(4)]

def kruskal_eps(df, group_col, value_col):
    groups = [g[value_col].values for _, g in df.groupby(group_col, observed=True)]
    h, p = sci_stats.kruskal(*groups)
    eps = (h - len(groups) + 1) / (len(df) - len(groups))
    return h, p, eps

panels = [
    ("foreign_population_pct", "#55A868", "Foreign-born population share\n\u2192 rate roughly doubles, step by step", axes[0]),
    ("second_vote_afd_pct", "#C44E52",
     "AfD vote share \u2192 lowest-AfD group has a higher rate,\nbut this is explained by pool count, not AfD itself", axes[1]),
]
for col, color, title, ax in panels:
    group_rates = analysis_df.groupby(f"{col}_quartile", observed=True)["racialised_rate"].median() * 100
    ax.bar(range(4), group_rates.values, color=color)
    ax.set_xticks(range(4))
    ax.set_xticklabels(labeled_ticks(col), fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("typical (median) racialised rate (%)")
    ax.tick_params(labelleft=True)  # keep y-axis numbers on both panels despite sharey (each must stand alone)
    h, p, eps = kruskal_eps(analysis_df, f"{col}_quartile", "racialised_rate")
    print(f"{col}: Kruskal-Wallis H={h:.1f}, p={p:.2e}, effect size (epsilon^2)={eps:.3f}")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

plt.tight_layout()
fig.savefig(f"{FIGDIR}/05_b1_b2_demographics.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the left panel (foreign-born population share) — B1:** constituencies in the
top quarter (17.3-33.1% foreign-born, 75 constituencies) have roughly **double** the rate
(1.9%) of the bottom quarter (3.2-10.0% foreign-born, 75 constituencies, rate 1.0%), with a
steady step up in between. **Yes, this is statistically significant** (p < 0.001, far below
any reasonable threshold), and it's the largest effect of anything tested against the rate
so far. It also **holds up** after separately checking it isn't just because those same
constituencies have fewer pools — it weakens somewhat once that's accounted for, but stays
real (confirmed again, more rigorously, in Part C). Benefits-recipient share and
unemployment show the same kind of pattern (not charted here — same method, same notebook).

**Reading the right panel (AfD vote share) — B2:** the lowest-AfD quarter (6.3-16.1% AfD,
75 constituencies) sits noticeably higher (1.6%) than the other three, which barely differ
from each other (~1.1-1.2%). This **is also statistically significant** (p = 0.007), but far
weaker than B1 — roughly a sixth of the effect size. And critically, **it does not survive**
the pool-count check: those low-AfD constituencies are disproportionately the same
higher-foreign-population, fewer-pool constituencies from B1, so once that overlap is
accounted for, the AfD-specific pattern disappears entirely. **A confound we caught and
ruled out, not a claim we're making.**

### Making B1 concrete — some actual constituencies

Quartile groups can feel abstract without seeing real places in them. A few from each end
of the foreign-born-population-share range:

In [ ]:
cols_to_show = ["constituency", "state", "foreign_population_pct", "racialised_rate"]
print("Highest foreign-born population share (Q4):")
print(analysis_df.nlargest(5, "foreign_population_pct")[cols_to_show]
      .assign(racialised_rate=lambda d: (d["racialised_rate"] * 100).round(2))
      .rename(columns={"foreign_population_pct": "foreign-born %", "racialised_rate": "racialised rate %"})
      .to_string(index=False))
print()
print("Lowest foreign-born population share (Q1):")
print(analysis_df.nsmallest(5, "foreign_population_pct")[cols_to_show]
      .assign(racialised_rate=lambda d: (d["racialised_rate"] * 100).round(2))
      .rename(columns={"foreign_population_pct": "foreign-born %", "racialised_rate": "racialised rate %"})
      .to_string(index=False))

**Reading it simply**: the high-foreign-population end is mostly big-city constituencies
(Frankfurt, Munich, Stuttgart, Mannheim) with rates mostly above the 1.5% national average;
the low end is mostly eastern/rural constituencies (Saxony, Saxony-Anhalt,
Mecklenburg-Vorpommern) with rates mostly below it. Not a perfect ladder for every single
constituency (e.g. `Offenbach` is high-foreign-population but a below-average rate), which
is exactly why we look at the group pattern (B1) rather than any one constituency.

---
# Part C — Does the signal from B1 survive a stricter check?

B1 and B2 each tested one factor at a time. Part C asks a harder question: if we account
for **every** factor at once — deprivation, pool count, age structure, vote shares,
turnout — together in a single model, rather than one at a time, does foreign-born
population share still matter?

**Yes — it's the one factor that survives every check we've run so far.**

- Alone, foreign-born population share tracks with a higher rate (B1).
- Still true after separately accounting for pool count.
- Still true — and still statistically clear — even after accounting for *everything else
  at once* in one combined model (deprivation, pool count, age, turnout, vote shares
  together).

Stated plainly, in a way that doesn't need any of the modelling machinery behind it: **each
additional percentage point of foreign-born population in a constituency is associated
with about a 3% higher racialised-review rate, even holding all the other factors fixed.**

Nothing else we tested (deprivation, turnout, individual party vote shares) survived being
tested alongside everything else this cleanly — most either weakened to nothing or turned
out to be too entangled with each other to separate (e.g. unemployment and benefits-receipt
measure much the same thing). Foreign-born population share is the one exception.

*(This is also the most preliminary of the three parts — the model is a first pass, still
leaves some of the geographic clustering from A2 unexplained, and is the piece most likely
to change on the next iteration.)*

---
# Research questions — what's answered, and what's next

*RQs revised 2026-07-14 after supervisor feedback on the first-pass presentation — the
original wording baked in assumptions the data hadn't earned yet (e.g. presupposing the
discourse "concentrates" or varies "systematically"). See `01_gmaps_feasibility.ipynb`'s
"Research questions" section for the same revision.*

**RQ1 — How is racialised discourse in pool reviews distributed across time and geography
in Germany?**
**Status: answered, strengthened since the presentation.** Part A holds: a real temporal
rise (A1), genuine geographic clustering confirmed statistically (A2), and a broad area
pattern rather than a venue artifact (A3). Since 2026-07-14: the timestamp window now
extends back to 2012 (every year from 2012 individually clears the notebook's volume bar,
so no bucketing was needed beyond a negligible pre-2012 pool) — 2012-2020 reads as flat/
noisy rather than a slow decline into the 2021 trough, so the 2021→2025 rise stands as a
real inflection, not the tail of a longer trend. A genuine seasonal pattern also emerged:
summer/outdoor-season reviews run 1.86% vs. 1.26% indoor-season (after correcting for a
timestamp-recovery artifact that would otherwise have piled estimated dates into one
month). The constituency reliability bar was tightened from `racialised_reviews>=5` to
`>=10` (247/299 constituencies now pass, down from 288/299) and a geometric
`is_border_constituency` flag was added (76/299 True) to help interpret the border-region
hotspots. A3's "not just one bad pool" result is recommended as an appendix item for the
paper — solid and paper-worthy, but a confound-ruling-out result rather than a new
substantive finding.

**RQ2 — Which constituency-level socio-economic and political characteristics are
independently associated with the racialised-review rate?**
**Status: first pass done, refined since the presentation.** Foreign-born population share
remains the one predictor that survives every check, including a joint model with
everything else accounted for at once. Since 2026-07-14: AfD vote share is now conclusively
a non-factor — its raw correlation (ρ=-0.204) moves to +0.032/+0.064 under every control
combination tried (venue density, foreign-population share, the new border flag, jointly) —
it was never an independent effect. A new `population_density_per_km2` column was added to
the pipeline and tested as an alternative to `pool_count`: density correlates strongly with
pool_count (ρ=-0.80) and edges it out in simple bivariate terms, but a proper multivariate
comparison (AIC/BIC/cross-validated MAE, with density log-transformed) shows `pool_count`
keeps a small, consistent edge — so it stays the primary control variable, with density
flagged as a close, theoretically-motivated alternative rather than a replacement. A new
caveat is now documented: 20 constituencies (Berlin, Hamburg, Bremen) share state-wide-
constant values for several predictor-ingredient columns (age structure, income, GDP,
benefits-receipt, unemployment) — not a bug, but these are estimates, not independently
observed constituency-level measurements; excluding them shifts some raw correlations by up
to 0.09 with no sign flips. Still open: untangling the two deprivation measures that are too
collinear to separate, and explaining the geographic clustering the model still doesn't
fully account for (Moran's I on residuals ≈0.11-0.12, about half the raw rate's 0.28).

**RQ3 — How does the content of racialised discourse (theme, framing, tone) differ across
places?**
**Status: partially started since the presentation.** A new notebook,
`01b_ethnicity_language.ipynb`, split the previously-conflated "Other/unknown" ethnicity
bucket into genuine `Unknown` (860 rows, missing prediction) vs. `Other` (2,427 rows, a real
known minority country), and added an `Arab` aggregate category (462 rows, 5.9% of
racialised reviews) for Arab-League countries that were previously scattered too thin to
see individually. Ethnicity and review language show a moderate association (Cramér's
V=0.239) that looks like genuine identity signal, not a mechanical artifact, for categories
with a corresponding language (e.g. France 49.7% French vs. a 3.1% corpus baseline, Poland
28.8% Polish, Russia 25.5% Russian). `theme`/`context`/`connotation` now have year-level
(and, sample-size permitting, month/season-level) breakdowns in `01_gmaps_feasibility.ipynb`
§1.4e, with explicit low-confidence caveats on `theme` (no human IAA yet) and `approach`
(near-chance-level human agreement, κ≈0-0.09) — still not yet combined with the new
ethnicity/language work above, or with the region/AfD-tercile crosstab that already exists in
`02_socioeconomic_political_relations.ipynb` §2.7. That combination is the natural next step
once `theme`/`approach` IAA is finalized.


## Where we're taking this next

- Part C's model still leaves some geographic clustering (A2) unexplained — worth digging
  into what that residual pattern is.
- Want to understand *why* the rate is rising over time (A1) — more reviews overall, more
  willingness to describe experiences this way, or a genuine change in what's happening at
  pools.
- Untangle the two deprivation measures (benefits-receipt, unemployment) that currently
  can't be separated from each other statistically.

## What this is *not*, yet

- Not causal — we can describe **where**, **when**, and **what tracks with what**, not yet **why**.
- Reviews are self-selected text, not a representative survey of everyone who visits a pool.
- Classification is automated (accuracy checked against human labels) — consistent across
  the corpus, but not perfect on any single review.
- Everything here is descriptive/correlational and first-pass — expected to keep changing
  as this iterates.

## Reflexivity check — what if we'd never seen the annotations?

*Added 2026-07-14 in response to a reviewer prompt: assume we never had access to the
`social`/`racialised`/`theme`/`approach`/`connotation`/`context` labels at all — what would
we have hypothesized, and what would we have done differently?*

- The obvious candidate predictors without any text classifier would still be **visible**
  correlates of intergroup contact/friction: local foreign-population share and diversity,
  political climate (turnout, vote shares) as a contact/conflict proxy, and urban/rural
  structure (density, venue crowding) — three of exactly the things Part B/C ended up
  testing anyway. That's mildly reassuring: the annotation pipeline converged on a similar
  predictor set to what a purely structural, no-text-reading hypothesis would have proposed.
- One thing we'd likely have done differently: use **review-language diversity itself**
  (which languages people write reviews in, at a place or constituency) as a cheap,
  unsupervised proxy for community diversity, available with zero classification work.
  `01b_ethnicity_language.ipynb`'s finding that language and predicted ethnicity correlate
  moderately (Cramér's V=0.239) suggests this proxy would have carried real signal.
- We'd also likely have designed in **temporal granularity and spatial modelling from the
  start** rather than adding them after a first pass — geographic clustering (Moran's I=0.28
  on the raw rate) and a genuine seasonal pattern (summer/outdoor higher) both turned out to
  be first-order features of the data, not nuisance variation to check for after the fact.
- What we would *not* have been able to get without the annotations: anything about RQ3 —
  the content/framing of racialised discourse has no visible-data proxy; that part
  necessarily depends on the classifier and its IAA.


## Discussion

- Does the regional clustering (A2) match anything you'd expect, or is it surprising?
- What's a defensible, non-causal way to talk about the foreign-population signal (B1/C)?
- What's the smallest next step that would make this feel ICWSM-ready to you?